# Spam IV: Logistic regression from scratch

Naive Bayes is *generative*: it models how each class produces words, and it counts correlated words twice. **Logistic
regression** is *discriminative*: it learns one weight per word directly by minimising the classification loss. The two are
closely related (both give a *linear* score), but LR usually wins when there is enough data.

We derive the model, write the loss in a **numerically stable** way, obtain the gradient with JAX (and check it by hand), train
it with two optimisers from the previous class (gradient descent and L-BFGS), and read the weights.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt
from scipy.optimize import minimize
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression

import spamlib as sl

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.dpi"] = 100

texts, y = sl.load()
X_txt_train, X_txt_test, y_train, y_test = sl.split(texts, y)
vocab = sl.build_vocab(X_txt_train)
index = {w: i for i, w in enumerate(vocab)}
tfidf = TfidfTransformer(sublinear_tf=True).fit(sl.count_matrix(X_txt_train, index))
X_train = jnp.asarray(tfidf.transform(sl.count_matrix(X_txt_train, index)).toarray())
X_test = jnp.asarray(tfidf.transform(sl.count_matrix(X_txt_test, index)).toarray())
y_tr = jnp.asarray(y_train, dtype=jnp.float64)
print(X_train.shape)

## 1. The model

A linear **score** (the *logit*, or log-odds) squashed into a probability by the **sigmoid**:

$$
z=\mathbf{w}^\top\mathbf{x}+b,\qquad P(\text{spam}\mid\mathbf{x})=\sigma(z)=\frac{1}{1+e^{-z}}
$$

Compare with Naive Bayes: its log-odds is $\log\frac{P(\text{spam})}{P(\text{ham})}+\sum_w x_w\log\frac{P(w\mid\text{spam})}{P(w\mid\text{ham})}$,
*also* a linear function of the counts. Same form, different way of choosing the weights.

## 2. The loss (negative log-likelihood, a.k.a. cross-entropy)

$$
J(\mathbf{w},b)=-\frac1n\sum_{i}\Big[y_i\log\sigma(z_i)+(1-y_i)\log\big(1-\sigma(z_i)\big)\Big]+\frac{\lambda}{2}\lVert\mathbf{w}\rVert^2
$$

### Stability trick 1: never compute `log(sigmoid(z))` naively

For a confidently wrong prediction $z=-800$: $\sigma(z)=e^{-800}$ underflows to exactly `0.0`, and $\log 0=-\infty$.
The identity $\log\sigma(z)=-\log(1+e^{-z})=-\operatorname{softplus}(-z)$ is computed without ever forming $\sigma(z)$, and
$\log(1-\sigma(z))=\log\sigma(-z)$.

In [ ]:
z = jnp.array([-800.0, -30.0, 0.0, 30.0, 800.0])
naive = jnp.log(1 / (1 + jnp.exp(-z)))
stable = jax.nn.log_sigmoid(z)
for zi, a, b in zip(z, naive, stable):
    print(f"z={float(zi):8.1f}   naive log(sigmoid) = {float(a):10.3f}   stable = {float(b):10.3f}")

In [ ]:
LAM = 1e-5


def loss(params, X, y, lam=LAM):
    w, b = params
    z = X @ w + b
    nll = -jnp.mean(y * jax.nn.log_sigmoid(z) + (1 - y) * jax.nn.log_sigmoid(-z))
    return nll + 0.5 * lam * jnp.dot(w, w)

## 3. The gradient

JAX differentiates the loss automatically. For logistic regression the analytic gradient is simple (and worth knowing):

$$
\nabla_{\mathbf{w}}J=\frac1n X^\top\big(\sigma(X\mathbf{w}+b)-\mathbf{y}\big)+\lambda\mathbf{w},\qquad
\partial_b J=\frac1n\sum_i\big(\sigma(z_i)-y_i\big)
$$

Always *verify* an autodiff or hand-written gradient against the other.

In [ ]:
params0 = (jnp.zeros(X_train.shape[1]), 0.0)
g_w, g_b = jax.grad(loss)(params0, X_train, y_tr)

resid = jax.nn.sigmoid(X_train @ params0[0] + params0[1]) - y_tr
g_w_hand = X_train.T @ resid / len(y_tr) + LAM * params0[0]
print("max |autodiff - analytic| =", float(jnp.abs(g_w - g_w_hand).max()))

## 4. Two optimisers, same loss

* **Gradient descent** $\;\theta\leftarrow\theta-\eta\nabla J\;$: simple, needs a learning rate $\eta$.
* **L-BFGS**: uses curvature information (a quasi-Newton method); no learning rate, far fewer iterations on smooth convex problems.

In [ ]:
loss_and_grad = jax.jit(jax.value_and_grad(loss), static_argnames=())


def gradient_descent(X, y, eta=50.0, steps=300, lam=LAM):
    params = (jnp.zeros(X.shape[1]), 0.0)
    history = []
    for _ in range(steps):
        value, g = loss_and_grad(params, X, y, lam)
        params = (params[0] - eta * g[0], params[1] - eta * g[1])
        history.append(float(value))
    return params, history


def lbfgs(X, y, lam=LAM, maxiter=200):
    d = X.shape[1]
    history = []

    def fun(theta):
        value, g = loss_and_grad((jnp.asarray(theta[:d]), theta[d]), X, y, lam)
        return float(value), np.concatenate([np.asarray(g[0]), [float(g[1])]])

    res = minimize(
        fun,
        np.zeros(d + 1),
        jac=True,
        method="L-BFGS-B",
        options={"maxiter": maxiter},
        callback=lambda th: history.append(fun(th)[0]),
    )
    return (jnp.asarray(res.x[:d]), float(res.x[d])), history


params_gd, hist_gd = gradient_descent(X_train, y_tr)
params_lb, hist_lb = lbfgs(X_train, y_tr)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(hist_gd, label="gradient descent (300 steps)")
ax.semilogy(hist_lb, label=f"L-BFGS ({len(hist_lb)} steps)")
ax.set(xlabel="iteration", ylabel="loss")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluation and comparison with scikit-learn

The three models minimise the same convex loss, so they should agree up to the stopping criteria.

In [ ]:
def evaluate(params, name):
    w, b = params
    score = np.asarray(X_test @ w + b)
    return name, sl.scores(y_test, (score >= 0).astype(int), score)


ref = LogisticRegression(C=1 / (LAM * len(y_train)), max_iter=2000).fit(np.asarray(X_train), y_train)
sk_score = ref.decision_function(np.asarray(X_test))
results = dict(
    [
        evaluate(params_gd, "LR gradient descent"),
        evaluate(params_lb, "LR L-BFGS"),
        ("LR scikit-learn", sl.scores(y_test, ref.predict(np.asarray(X_test)), sk_score)),
    ]
)
sl.show(results)

## 6. Read the weights: the gradient *is* the attack direction

The score is linear, so $\partial z/\partial x_j=w_j$: **the weights are the gradient of the decision with respect to the
input**. An attacker with white-box access reads off, for each word, how much adding it moves the score, and picks the words
with the most negative $w_j$. This is the linear special case of the gradient-based attacks (FGSM) from the first class.

In [ ]:
w = np.asarray(params_lb[0])
order = np.argsort(w)
print("pushes to HAM :", [(vocab[i], round(float(w[i]), 1)) for i in order[:12]])
print("pushes to SPAM:", [(vocab[i], round(float(w[i]), 1)) for i in order[::-1][:12]])

## 7. Regularisation and the decision threshold

$\lambda$ trades fit for simplicity: with $\lambda$ too small the model memorises rare words; too large, it ignores them.
For a spam filter the threshold matters more than $\lambda$: **raise it until precision is acceptable**, then read the recall.

In [ ]:
for lam in (1e-6, 1e-5, 1e-4, 1e-3):
    p, _ = lbfgs(X_train, y_tr, lam=lam)
    _, r = evaluate(p, f"lambda={lam:.0e}")
    print(f"lambda={lam:.0e}  precision={r['precision']:.3f}  recall={r['recall']:.3f}  f1={r['f1']:.3f}")

Now the threshold: at a fixed model, moving the decision threshold on the score trades recall for precision.

In [ ]:
w_, b_ = params_lb
score = np.asarray(X_test @ w_ + b_)
for t in (-1.0, -0.5, 0.0, 1.0, 2.0):
    r = sl.scores(y_test, (score >= t).astype(int), score)
    print(f"threshold z>={t:+.1f}  precision={r['precision']:.3f}  recall={r['recall']:.3f}")

## Exercises

1. Remove the L2 term (`lam=0`) and train on 200 messages only. What happens to the weight of rare words?
2. Replace the fixed learning rate by a decaying one, $\eta_t=\eta_0/(1+0.01\,t)$. Does it converge faster?
3. Add class weights (spam counts 5x) to the loss. What happens to precision and recall at threshold 0?
4. **Attacker.** Take a spam message from the test set, and greedily append the ham-pushing word with the most negative weight
   until the score crosses 0. How many words do you need? Compare with Naive Bayes (notebook 07 automates this).